In [1]:
!pip install open_clip_torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.5 MB ? eta -:--:--

     ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/1.5 MB 8.1 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.9 MB/s eta 0:00:00


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import cv2
import torch
import open_clip
import csv
import numpy as np
from PIL import Image
import torch_xla.core.xla_model as xm

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/usr/local/lib/python3.10/site-packages/torch_xla/__init__.py:202: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


In [3]:
sub_images_dir = "/kaggle/input/code-code-code"
output_dir = "/kaggle/working/videos"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [4]:
def preprocess_sub_images(sub_images):
    preprocessed_sub_images = []
    for sub_image_path in sub_images:
        image = Image.open(sub_image_path).convert('RGB')
        preprocessed_image = preprocess_val(image).unsqueeze(0).to(device)
        preprocessed_sub_images.append(preprocessed_image)
    return preprocessed_sub_images

def process_sub_images_in_batches(sub_images, batch_size=16):
    features = []
    for i in range(0, len(sub_images), batch_size):
        batch_sub_images = sub_images[i:i+batch_size]
        batch_sub_images_tensor = torch.cat(batch_sub_images)
        with torch.no_grad():
            batch_features = model.encode_image(batch_sub_images_tensor)
            features.append(batch_features.cpu().numpy())
        del batch_sub_images_tensor
        xm.mark_step()
    return np.concatenate(features, axis=0)

In [5]:
device = xm.xla_device()
print(f"Using device: {device}")
model_name = 'hf-hub:laion/CLIP-ViT-H-14-laion2B-s32B-b79K'
model, preprocess_val, preprocess_train = open_clip.create_model_and_transforms(model_name)
model = model.to(device)
tokenizer = open_clip.get_tokenizer(model_name)
print("OpenCLIP model loaded")

Using device: xla:0


E0000 00:00:1729004974.340146      77 common_lib.cc:818] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:483


/usr/local/lib/python3.10/site-packages/open_clip/factory.py:129: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=map_loc

OpenCLIP model loaded


In [6]:
feature_dir = os.path.join(output_dir, 'feature')
metadata_dir = os.path.join(output_dir, 'object_metadata')
os.makedirs(feature_dir, exist_ok=True)
os.makedirs(metadata_dir, exist_ok=True)

for video_folder in os.listdir(sub_images_dir):
    #if video_folder.startswith('L01') or video_folder.startswith('L02'):
    video_folder_path = os.path.join(sub_images_dir, video_folder)
    if os.path.isdir(video_folder_path):
        print(f"Processing video folder: {video_folder}")
        
        sub_images = sorted(
        [os.path.join(video_folder_path, img) for img in os.listdir(video_folder_path) if img.endswith('.jpg')]
        )

        preprocessed_sub_images = preprocess_sub_images(sub_images)

        features = process_sub_images_in_batches(preprocessed_sub_images)
            
        all_combined_features = []
        all_metadata = []
            
        for image_path, feature in zip(sub_images, features):
            filename = os.path.basename(image_path).replace('.jpg', '')
            parts = filename.split('_')
                
            video_id = '_'.join(parts[:2])
            keyframe = parts[2]
            object_idx = parts[3]
                
            combined_feature = feature / np.linalg.norm(feature)  # Normalize
            all_combined_features.append(combined_feature)
            all_metadata.append((video_id, keyframe, object_idx))  # Store metadata for this object
            
        all_combined_features = np.array(all_combined_features).astype('float32')
            
        output_path = os.path.join(feature_dir, f"{video_folder}.npy")
        np.save(output_path, features)
        print(f"Saved features for {video_folder} to {output_path}")
            
        metadata_file_path = os.path.join(metadata_dir, f"{video_folder}_metadata.csv")
        with open(metadata_file_path, 'w', newline='') as f:
            csv_writer = csv.writer(f)
            csv_writer.writerow(['video_id', 'keyframe', 'object_idx'])  # Write header
            csv_writer.writerows(all_metadata)  # Write metadata rows

print("Feature extraction and saving completed")

Processing video folder: L26_V383


Saved features for L26_V383 to /kaggle/working/videos/feature/L26_V383.npy
Processing video folder: L26_V361


Saved features for L26_V361 to /kaggle/working/videos/feature/L26_V361.npy
Processing video folder: L26_V360


Saved features for L26_V360 to /kaggle/working/videos/feature/L26_V360.npy
Processing video folder: L26_V311


Saved features for L26_V311 to /kaggle/working/videos/feature/L26_V311.npy
Processing video folder: L26_V320


...

Saved features for L26_V378 to /kaggle/working/videos/feature/L26_V378.npy
Processing video folder: L26_V385


Saved features for L26_V385 to /kaggle/working/videos/feature/L26_V385.npy
Processing video folder: L26_V399


Saved features for L26_V399 to /kaggle/working/videos/feature/L26_V399.npy
Processing video folder: L26_V324


Saved features for L26_V324 to /kaggle/working/videos/feature/L26_V324.npy
Processing video folder: L26_V365


Saved features for L26_V365 to /kaggle/working/videos/feature/L26_V365.npy
Feature extraction and saving completed
